# PSO para reglas dominantes basadas en similaridad

Este notebook utiliza optimización por enjambre de partículas (PSO) para descubrir reglas dominantes:

$$
X \Rightarrow Y
$$

El antecedente \(X\) puede combinar condiciones continuas evaluadas mediante similaridad y condiciones categóricas evaluadas por igualdad exacta. El consecuente \(Y\) corresponde a la presencia o ausencia de enfermedad cardiovascular.

## 1. Carga de datos

In [ ]:
import copy
import gc
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# CARGAR DATASET NORMALIZADO Y METADATOS
# ============================================================

# 1. Conectar repositorio local
# 2. Definir carpeta de origen
from pathlib import Path


def localizar_raiz_repositorio():
    """Encuentra la carpeta que contiene notebooks/ del repositorio."""
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "notebooks").is_dir():
            return candidato
    return Path.cwd()


raiz_repositorio = localizar_raiz_repositorio()
carpeta_origen = raiz_repositorio / "artifacts" / "preprocesamiento"

# 3. Cargar el dataset normalizado que utilizará PSO
df_similitud = pd.read_csv(carpeta_origen / "cardiovascular_normalizado_robusto.csv")

# 4. Cargar los parámetros de normalización y los mapas
metadatos_normalizacion = joblib.load(carpeta_origen / "metadatos_normalizacion_robusta.pkl")


# 5. Recuperar la información guardada
parametros_normalizacion_robusta = metadatos_normalizacion["parametros_normalizacion"]
mapas_categorias = metadatos_normalizacion["mapas_categorias"]
columnas_continuas = metadatos_normalizacion["columnas_continuas"]
columnas_categoricas = metadatos_normalizacion["columnas_categoricas"]
columna_objetivo = metadatos_normalizacion["columna_objetivo"]


# 6. Mostrar confirmación
print(f"Dataset cargado con forma: {df_similitud.shape}")
print(f"\nVariables continuas: {columnas_continuas}")
print(f"\nVariables categóricas: {columnas_categoricas}")
print(f"\nVariable objetivo: {columna_objetivo}")

In [ ]:
# ============================================================
# VALIDAR Y SEPARAR EL ESPACIO DE BÚSQUEDA
# ============================================================

# 1. Definir las variables candidatas para el antecedente X
columnas_antecedente = columnas_continuas + columnas_categoricas
columnas_requeridas = columnas_antecedente + [columna_objetivo]


# 2. Verificar que todas las columnas existan
columnas_faltantes = [
    columna for columna in columnas_requeridas
    if columna not in df_similitud.columns
]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas necesarias: {columnas_faltantes}")


# 3. Verificar que no existan columnas repetidas
if len(columnas_antecedente) != len(set(columnas_antecedente)):
    raise ValueError("Existen columnas repetidas en el antecedente.")

if columna_objetivo in columnas_antecedente:
    raise ValueError("La variable objetivo no puede formar parte del antecedente.")


# 4. Separar antecedentes y consecuente
X_datos = df_similitud[columnas_antecedente].copy()
y_objetivo = df_similitud[columna_objetivo].copy()


# 5. Validar las variables continuas
for columna in columnas_continuas:
    if not pd.api.types.is_numeric_dtype(X_datos[columna]):
        raise TypeError(f"La variable continua '{columna}' no es numérica.")

    minimo = X_datos[columna].min()
    maximo = X_datos[columna].max()

    if minimo < 0 or maximo > 1:
        raise ValueError(
            f"'{columna}' contiene valores fuera de [0, 1]: "
            f"mínimo={minimo}, máximo={maximo}"
        )


# 6. Recuperar los valores válidos de las variables categóricas
valores_categoricos = {
    columna: sorted(X_datos[columna].dropna().unique().tolist())
    for columna in columnas_categoricas
}


# 7. Validar el consecuente
valores_objetivo = sorted(y_objetivo.dropna().unique().tolist())

if not set(valores_objetivo).issubset({0, 1}):
    raise ValueError(
        "La variable objetivo debe contener solamente 0 y 1. "
        f"Valores encontrados: {valores_objetivo}")

y_objetivo = y_objetivo.astype(np.int8)


# 8. Verificar valores faltantes
faltantes_antecedente = int(X_datos.isna().sum().sum())
faltantes_objetivo = int(y_objetivo.isna().sum())


# 9. Calcular la distribución del consecuente
distribucion_objetivo = (
    y_objetivo.value_counts()
    .sort_index()
    .rename_axis("clase")
    .reset_index(name="cantidad")
)

distribucion_objetivo["proporcion"] = (
    distribucion_objetivo["cantidad"] / len(y_objetivo)
)


# 10. Mostrar confirmación
print(f"Espacio de búsqueda: {X_datos.shape}")
print(f"Variables candidatas para X: {len(columnas_antecedente)}")
print(f"Valores faltantes en X: {faltantes_antecedente}")
print(f"Valores faltantes en Y: {faltantes_objetivo}")

print(f"\nValores categóricos: {valores_categoricos}")
print("\nDistribución del consecuente:")

display(distribucion_objetivo)

### Ejemplo para revertir valores normalizados


In [ ]:
# ============================================================
# FUNCIONES DE SIMILARIDAD
# ============================================================

# 1. Similaridad triangular
def similitud_triangular(x, centro, radio):
    """Calcula una similaridad triangular entre 0 y 1."""
    if radio <= 0:
        raise ValueError("El radio debe ser mayor que cero.")

    x = np.asarray(x, dtype=float)
    return np.maximum(0, 1 - np.abs(x - centro) / radio)


# 2. Similaridad gaussiana truncada
def similitud_gaussiana_truncada(x, centro, radio):
    """Calcula una similaridad gaussiana con sigma = radio / 3."""
    if radio <= 0:
        raise ValueError("El radio debe ser mayor que cero.")

    x = np.asarray(x, dtype=float)
    sigma = radio / 3
    distancia = np.abs(x - centro)
    similitud = np.exp(-0.5 * (distancia / sigma) ** 2)

    dentro_radio = (
        (distancia < radio)
        & ~np.isclose(distancia, radio, rtol=1e-10, atol=1e-12)
    )

    return np.where(dentro_radio, similitud, 0.0)


# 3. Activación categórica por igualdad exacta
def activacion_categorica(valores, valor_buscado):
    """Devuelve 1 cuando la categoría coincide y 0 en otro caso."""
    return (np.asarray(valores) == valor_buscado).astype(float)


# 4. Comparación visual
centro_ejemplo = 0.50
radio_ejemplo = 0.30
x_ejemplo = np.linspace(0, 1, 500)

plt.figure(figsize=(9, 4))
plt.plot(
    x_ejemplo,
    similitud_triangular(x_ejemplo, centro_ejemplo, radio_ejemplo),
    label="Triangular"
)
plt.plot(
    x_ejemplo,
    similitud_gaussiana_truncada(x_ejemplo, centro_ejemplo, radio_ejemplo),
    label="Gaussiana truncada"
)
plt.axvline(centro_ejemplo, linestyle="--", color="gray", label="Centro")
plt.axvline(centro_ejemplo - radio_ejemplo, linestyle=":", color="gray")
plt.axvline(centro_ejemplo + radio_ejemplo, linestyle=":", color="gray")

plt.xlabel("Valor normalizado")
plt.ylabel("Similaridad")
plt.title("Funciones de similaridad")
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


El objetivo de este bloque es compactar el código de las funciones de similaridad para mejorar la concisión.

In [ ]:
# ============================================================
# CALCULAR LA ACTIVACIÓN DEL ANTECEDENTE
# ============================================================

def calcular_activacion_antecedente(
    dataframe,
    condiciones,
    metodo_similitud="triangular",
    devolver_detalle=False
):
    """Combina las condiciones del antecedente mediante el mínimo."""

    # 1. Validaciones generales
    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError("'dataframe' debe ser un pandas.DataFrame.")

    if not isinstance(condiciones, list) or not condiciones:
        raise ValueError("'condiciones' debe ser una lista no vacía.")

    if metodo_similitud not in {"triangular", "gaussiana"}:
        raise ValueError(
            "'metodo_similitud' debe ser 'triangular' o 'gaussiana'."
        )

    if any(not isinstance(condicion, dict) for condicion in condiciones):
        raise TypeError("Cada condición debe ser un diccionario.")


    # 2. Verificar las variables del antecedente
    variables = [condicion.get("variable") for condicion in condiciones]

    if None in variables:
        raise ValueError("Todas las condiciones deben incluir una variable.")

    if len(variables) != len(set(variables)):
        raise ValueError("Una variable no puede aparecer más de una vez.")


    # 3. Calcular las activaciones individuales
    activaciones = []
    nombres = []

    for condicion in condiciones:
        variable = condicion["variable"]
        tipo = condicion.get("tipo")

        if variable not in dataframe.columns:
            raise ValueError(f"La variable '{variable}' no existe.")

        if tipo == "continua":
            if variable not in columnas_continuas:
                raise ValueError(f"'{variable}' no está registrada como continua.")

            if "centro" not in condicion or "radio" not in condicion:
                raise ValueError(
                    f"La condición continua '{variable}' necesita centro y radio."
                )

            centro = float(condicion["centro"])
            radio = float(condicion["radio"])

            if not 0 <= centro <= 1:
                raise ValueError(f"El centro de '{variable}' debe estar en [0, 1].")

            if radio <= 0:
                raise ValueError(f"El radio de '{variable}' debe ser positivo.")

            funcion_similitud = (
                similitud_triangular
                if metodo_similitud == "triangular"
                else similitud_gaussiana_truncada
            )

            activacion = funcion_similitud(
                dataframe[variable].to_numpy(),
                centro,
                radio
            )
            nombre = f"{variable}_c={centro:.3f}_r={radio:.3f}"

        elif tipo == "categorica":
            if variable not in columnas_categoricas:
                raise ValueError(f"'{variable}' no está registrada como categórica.")

            if "valor" not in condicion:
                raise ValueError(
                    f"La condición categórica '{variable}' necesita un valor."
                )

            valor = condicion["valor"]

            if valor not in valores_categoricos[variable]:
                raise ValueError(
                    f"Valor inválido para '{variable}': {valor}. "
                    f"Dominio: {valores_categoricos[variable]}"
                )

            activacion = activacion_categorica(
                dataframe[variable].to_numpy(),
                valor
            )
            nombre = f"{variable}={valor}"

        else:
            raise ValueError(
                f"El tipo de '{variable}' debe ser 'continua' o 'categorica'."
            )

        activaciones.append(activacion)
        nombres.append(nombre)


    # 4. Combinar las condiciones con la t-norma mínimo
    matriz_activaciones = np.vstack(activaciones)
    activacion_antecedente = np.min(matriz_activaciones, axis=0)


    # 5. Devolver el detalle cuando sea solicitado
    if devolver_detalle:
        detalle = pd.DataFrame(
            matriz_activaciones.T,
            columns=nombres,
            index=dataframe.index
        )
        detalle["activacion_antecedente"] = activacion_antecedente
        return activacion_antecedente, detalle

    return activacion_antecedente

## 5. Métricas de una regla

Después de calcular la activación del antecedente \(X\), evaluamos qué tan buena es la regla:

$$
X \Rightarrow Y
$$

Cada registro puede cumplir parcialmente el antecedente. Por eso, las métricas se calculan usando los grados de activación.

### Soporte del antecedente

El soporte indica qué tanto aparece el antecedente en el dataset:

$$
Supp(X)=
\frac{1}{N}
\sum_{i=1}^{N}\mu_X(o_i)
$$

### Soporte de la regla

El soporte de la regla considera que se cumplan simultáneamente el antecedente y el consecuente:

$$
Supp(X\land Y)=
\frac{1}{N}
\sum_{i=1}^{N}
\min\left(\mu_X(o_i),\mu_Y(o_i)\right)
$$

Como el consecuente es categórico, su activación es exacta:

$$
\mu_Y(o_i)=
\begin{cases}
1, & \text{si el registro pertenece a la clase buscada} \\
0, & \text{en otro caso}
\end{cases}
$$

### Confianza

La confianza responde a la pregunta:

> De los registros que cumplen el antecedente, ¿cuántos pertenecen también al consecuente?

Se calcula como:

$$
Conf(X\Rightarrow Y)=
\frac{Supp(X\land Y)}
{Supp(X)}
$$

Una confianza de \(0.70\) significa que aproximadamente el 70 % de la activación del antecedente se relaciona con el consecuente.

### Lift

El lift compara la confianza de la regla con la frecuencia general del consecuente:

$$
Lift(X\Rightarrow Y)=
\frac{Conf(X\Rightarrow Y)}
{Supp(Y)}
$$

Su interpretación general es:

- \(Lift>1\): asociación positiva.
- \(Lift=1\): comportamiento similar a la frecuencia general.
- \(Lift<1\): asociación negativa.

Estas métricas serán utilizadas posteriormente por la función fitness para orientar el movimiento de PSO.

In [ ]:
# ============================================================
# MÉTRICAS DE UNA REGLA
# ============================================================

def calcular_metricas_regla(
    activacion_antecedente,
    objetivo,
    valor_consecuente):
    """Calcula soporte, confianza y lift de X => Y."""

    activacion_X = np.asarray(
        activacion_antecedente,
        dtype=float
    )

    objetivo = np.asarray(objetivo)

    if len(activacion_X) != len(objetivo):
        raise ValueError(
            "La activación y el objetivo deben tener la misma longitud."
        )

    if valor_consecuente not in {0, 1}:
        raise ValueError(
            "'valor_consecuente' debe ser 0 o 1."
        )

    if np.any((activacion_X < 0) | (activacion_X > 1)):
        raise ValueError(
            "Las activaciones deben estar dentro de [0, 1]."
        )


    # 1. Activación del consecuente
    activacion_Y = (
        objetivo == valor_consecuente
    ).astype(float)


    # 2. Activación conjunta X AND Y
    activacion_XY = np.minimum(
        activacion_X,
        activacion_Y
    )


    # 3. Soportes
    soporte_X = float(np.mean(activacion_X))
    soporte_Y = float(np.mean(activacion_Y))
    soporte_XY = float(np.mean(activacion_XY))


    # 4. Confianza
    confianza = (
        soporte_XY / soporte_X
        if soporte_X > 0
        else 0.0
    )


    # 5. Lift
    lift = (
        confianza / soporte_Y
        if soporte_Y > 0
        else 0.0
    )


    # 6. Cantidades descriptivas
    return {
        "valor_consecuente": int(valor_consecuente),
        "soporte_X": soporte_X,
        "soporte_Y": soporte_Y,
        "soporte_XY": soporte_XY,
        "confianza": float(confianza),
        "lift": float(lift),
        "registros_X_positivos": int(np.sum(activacion_X > 0)),
        "registros_XY_positivos": int(np.sum(activacion_XY > 0)),
        "suma_activacion_X": float(np.sum(activacion_X)),
        "suma_activacion_XY": float(np.sum(activacion_XY))
    }


## 6. Factor de certeza

El factor de certeza compara la confianza de una regla con la frecuencia general del consecuente.

Permite saber si el antecedente aumenta, mantiene o reduce la presencia esperada de la clase.

Cuando la confianza es mayor que la frecuencia base:

$$
CF=
\frac{Conf(X\Rightarrow Y)-Supp(Y)}
{1-Supp(Y)}
$$

Cuando la confianza es menor que la frecuencia base:

$$
CF=
\frac{Conf(X\Rightarrow Y)-Supp(Y)}
{Supp(Y)}
$$

Su interpretación es:

- \(CF>0\): el antecedente favorece al consecuente.
- \(CF=0\): el antecedente no cambia la frecuencia esperada.
- \(CF<0\): el antecedente reduce la presencia del consecuente.

Para las reglas dominantes nos interesan principalmente los valores positivos.

In [ ]:
# ============================================================
# FACTOR DE CERTEZA
# ============================================================

def calcular_factor_certeza(
    confianza,
    soporte_consecuente
):
    """Calcula el factor de certeza de X => Y."""

    if not 0 <= confianza <= 1:
        raise ValueError("'confianza' debe estar entre 0 y 1.")

    if not 0 <= soporte_consecuente <= 1:
        raise ValueError(
            "'soporte_consecuente' debe estar entre 0 y 1."
        )

    if soporte_consecuente == 0:
        return 0.0 if confianza == 0 else 1.0

    if soporte_consecuente == 1:
        return 0.0 if confianza == 1 else -1.0

    if confianza > soporte_consecuente:
        factor_certeza = (
            (confianza - soporte_consecuente)
            / (1 - soporte_consecuente)
        )
    elif confianza < soporte_consecuente:
        factor_certeza = (
            (confianza - soporte_consecuente)
            / soporte_consecuente
        )
    else:
        factor_certeza = 0.0

    return float(factor_certeza)

## 7. Evaluación completa de una regla

Hasta ahora calculamos por separado:

1. La activación del antecedente.
2. El soporte.
3. La confianza.
4. El lift.
5. El factor de certeza.

Esta función integra todos esos cálculos para evaluar una regla completa:

$$
X \Rightarrow Y
$$

El resultado será utilizado posteriormente por la función fitness y por PSO.

In [ ]:
# ============================================================
# EVALUACIÓN COMPLETA DE UNA REGLA
# ============================================================

def evaluar_regla_completa(
    dataframe,
    objetivo,
    condiciones,
    valor_consecuente,
    metodo_similitud="triangular"
):
    """Evalúa una regla y devuelve todas sus métricas."""

    activacion_X = calcular_activacion_antecedente(
        dataframe=dataframe,
        condiciones=condiciones,
        metodo_similitud=metodo_similitud
    )

    metricas = calcular_metricas_regla(
        activacion_antecedente=activacion_X,
        objetivo=objetivo,
        valor_consecuente=valor_consecuente
    )

    metricas["factor_certeza"] = calcular_factor_certeza(
        confianza=metricas["confianza"],
        soporte_consecuente=metricas["soporte_Y"]
    )

    metricas["condiciones"] = condiciones
    metricas["numero_condiciones"] = len(condiciones)
    metricas["metodo_similitud"] = metodo_similitud
    metricas["activacion_antecedente"] = activacion_X

    return metricas

## 4. Funciones de ayuda para la traducción de reglas

Estas funciones son necesarias para convertir valores normalizados a sus unidades originales.

## 8. Representación de una partícula

Cada partícula representa una regla candidata:

$$
X \Rightarrow Y
$$

La partícula contiene dos grupos de variables:

- Variables continuas: centro y radio.
- Variables categóricas: valor elegido.

Cada variable también tiene un estado:

$$
activa \in \{0,1\}
$$

Cuando una variable está inactiva, no forma parte del antecedente. El antecedente debe contener entre una y cuatro condiciones.

Las variables continuas trabajan en el espacio normalizado \([0,1]\). Las categóricas solo pueden utilizar valores presentes en el dataset.

In [ ]:
# ============================================================
# REPRESENTACIÓN DE UNA PARTÍCULA
# ============================================================

MIN_CONDICIONES = 1
MAX_CONDICIONES = 4

LIMITE_CENTRO_INFERIOR = 0.0
LIMITE_CENTRO_SUPERIOR = 1.0

RADIO_MINIMO = 0.02
RADIO_MAXIMO = 0.30


# 1. Crear una partícula vacía
def crear_particula_vacia():
    """Crea una partícula sin variables activas."""

    particula = {
        "continuas": {},
        "categoricas": {}
    }

    for variable in columnas_continuas:
        particula["continuas"][variable] = {
            "activa": 0,
            "centro": 0.5,
            "radio": 0.1
        }

    for variable in columnas_categoricas:
        particula["categoricas"][variable] = {
            "activa": 0,
            "valor": valores_categoricos[variable][0]
        }

    return particula


# 2. Contar condiciones activas
def contar_condiciones_activas(particula):
    """Cuenta las variables activas del antecedente."""

    continuas_activas = sum(
        dato["activa"]
        for dato in particula["continuas"].values()
    )

    categoricas_activas = sum(
        dato["activa"]
        for dato in particula["categoricas"].values()
    )

    return int(continuas_activas + categoricas_activas)


# 3. Convertir una partícula en condiciones
def decodificar_particula(particula):
    """Convierte una partícula en una lista de condiciones."""

    condiciones = []

    for variable, dato in particula["continuas"].items():
        if dato["activa"] == 1:
            condiciones.append({
                "variable": variable,
                "tipo": "continua",
                "centro": float(dato["centro"]),
                "radio": float(dato["radio"])
            })

    for variable, dato in particula["categoricas"].items():
        if dato["activa"] == 1:
            condiciones.append({
                "variable": variable,
                "tipo": "categorica",
                "valor": dato["valor"]
            })

    return condiciones

## 9. Función fitness para reglas dominantes

La función fitness indica qué tan conveniente es una regla para PSO.

La propuesta es:

$$
Fitness_{dom}=
P_{soporte}
\cdot
Confianza
\cdot
CF^{+}
\cdot
P_{longitud}
$$

donde:

$$
P_{soporte}=
\min\left(
1,
\frac{Supp(X\land Y)}
{minSupp_{dom}}
\right)
$$

$$
CF^{+}=\max(0,CF)
$$

$$
P_{longitud}=
\frac{1}{1+|X|}
$$

La función favorece reglas con soporte suficiente, confianza alta, asociación positiva y pocos atributos.

In [ ]:
# ============================================================
# FITNESS DE REGLAS DOMINANTES
# ============================================================

def calcular_fitness_dominante(
    metricas_regla,
    soporte_minimo,
    max_condiciones=4
):
    """Calcula el fitness de una regla dominante."""

    soporte_regla = float(metricas_regla["soporte_XY"])
    confianza = float(metricas_regla["confianza"])
    factor_certeza = float(metricas_regla["factor_certeza"])
    numero_condiciones = int(
        metricas_regla["numero_condiciones"]
    )

    if soporte_minimo <= 0:
        raise ValueError(
            "'soporte_minimo' debe ser mayor que cero."
        )

    if numero_condiciones < 1:
        raise ValueError(
            "La regla debe tener al menos una condición."
        )

    if numero_condiciones > max_condiciones:
        return {
            "fitness": 0.0,
            "penalizacion_soporte": 0.0,
            "confianza": confianza,
            "factor_certeza_positivo": 0.0,
            "penalizacion_longitud": 0.0,
            "regla_valida": False
        }

    penalizacion_soporte = min(
        1.0,
        soporte_regla / soporte_minimo
    )

    factor_certeza_positivo = max(
        0.0,
        factor_certeza
    )

    penalizacion_longitud = (
        1 / (1 + numero_condiciones)
    )

    fitness = (
        penalizacion_soporte
        * confianza
        * factor_certeza_positivo
        * penalizacion_longitud
    )

    return {
        "fitness": float(fitness),
        "penalizacion_soporte": float(penalizacion_soporte),
        "confianza": float(confianza),
        "factor_certeza_positivo": float(
            factor_certeza_positivo
        ),
        "penalizacion_longitud": float(
            penalizacion_longitud
        ),
        "regla_valida": True
    }

## 10. Criterio para aceptar una regla dominante

La función fitness orienta el movimiento de PSO, pero no determina por sí sola si una regla será aceptada como dominante.

Una regla se considera dominante cuando cumple simultáneamente:

$$
Supp(X\land Y)\geq minSupp_{dom}
$$

$$
Conf(X\Rightarrow Y)\geq minConf_{dom}
$$

$$
Lift(X\Rightarrow Y)>minLift_{dom}
$$

$$
CF(X\Rightarrow Y)\geq minCF_{dom}
$$

Además, debe contener entre una y cuatro condiciones.

Estos umbrales son criterios de aceptación y pueden modificarse posteriormente para realizar análisis de sensibilidad.

In [ ]:
# ============================================================
# CRITERIO DE DOMINANCIA
# ============================================================

def verificar_regla_dominante(
    resultado_regla,
    soporte_minimo=0.005,
    confianza_minima=0.60,
    lift_minimo=1.00,
    factor_certeza_minimo=0.10,
    min_condiciones=1,
    max_condiciones=4
):
    """Verifica si una regla cumple los criterios de dominancia."""

    metricas = resultado_regla["metricas"]

    criterios = {
        "fitness_positivo": resultado_regla["fitness"] > 0,
        "soporte_suficiente": (
            metricas["soporte_XY"] >= soporte_minimo
        ),
        "confianza_suficiente": (
            metricas["confianza"] >= confianza_minima
        ),
        "lift_suficiente": (
            metricas["lift"] > lift_minimo
        ),
        "factor_certeza_suficiente": (
            metricas["factor_certeza"]
            >= factor_certeza_minimo
        ),
        "longitud_valida": (
            min_condiciones
            <= metricas["numero_condiciones"]
            <= max_condiciones
        )
    }

    criterios_incumplidos = [
        nombre
        for nombre, cumple in criterios.items()
        if not cumple
    ]

    return {
        "es_dominante": all(criterios.values()),
        "criterios": criterios,
        "criterios_incumplidos": criterios_incumplidos,
        "umbrales": {
            "soporte_minimo": soporte_minimo,
            "confianza_minima": confianza_minima,
            "lift_minimo": lift_minimo,
            "factor_certeza_minimo": factor_certeza_minimo,
            "min_condiciones": min_condiciones,
            "max_condiciones": max_condiciones
        }
    }

## 11. Evaluar una partícula individual

Antes de ejecutar todo el enjambre, probaremos una sola partícula.

La partícula se convertirá en una regla:

$$
X \Rightarrow Y
$$

Después se calcularán sus métricas, su fitness y se verificará si cumple los criterios para ser considerada dominante.

In [ ]:
# ============================================================
# EVALUAR UNA PARTÍCULA INDIVIDUAL
# ============================================================

def evaluar_particula_dominante(
    particula,
    dataframe,
    objetivo,
    valor_consecuente,
    metodo_similitud,
    soporte_minimo,
    max_condiciones
):
    """Convierte una partícula en regla y calcula su fitness."""

    condiciones = decodificar_particula(particula)

    metricas = evaluar_regla_completa(
        dataframe=dataframe,
        objetivo=objetivo,
        condiciones=condiciones,
        valor_consecuente=valor_consecuente,
        metodo_similitud=metodo_similitud
    )

    componentes_fitness = calcular_fitness_dominante(
        metricas_regla=metricas,
        soporte_minimo=soporte_minimo,
        max_condiciones=max_condiciones
    )

    return {
        "condiciones": condiciones,
        "metricas": metricas,
        "componentes_fitness": componentes_fitness,
        "fitness": componentes_fitness["fitness"]
    }


# ============================================================
# PRUEBA CON UNA PARTÍCULA
# ============================================================

particula_prueba = crear_particula_vacia()

particula_prueba["continuas"]["edad_anios"] = {
    "activa": 1,
    "centro": 0.60,
    "radio": 0.15
}

particula_prueba["categoricas"]["sexo"] = {
    "activa": 1,
    "valor": 2
}

particula_prueba["categoricas"]["colesterol"] = {
    "activa": 1,
    "valor": 3
}


resultado_particula_prueba = evaluar_particula_dominante(
    particula=particula_prueba,
    dataframe=X_datos,
    objetivo=y_objetivo,
    valor_consecuente=1,
    metodo_similitud="triangular",
    soporte_minimo=0.005,
    max_condiciones=MAX_CONDICIONES
)

validacion_particula = verificar_regla_dominante(
    resultado_regla=resultado_particula_prueba,
    soporte_minimo=0.005,
    confianza_minima=0.60,
    lift_minimo=1.00,
    factor_certeza_minimo=0.10,
    min_condiciones=MIN_CONDICIONES,
    max_condiciones=MAX_CONDICIONES
)

# ============================================================
# MOSTRAR RESULTADO
# ============================================================

metricas_prueba = resultado_particula_prueba["metricas"]

print("Regla evaluada:")

for condicion in resultado_particula_prueba["condiciones"]:
    print(f"  {condicion}")

print(f"\nFitness: {resultado_particula_prueba['fitness']:.8f}")
print(f"Soporte X: {metricas_prueba['soporte_X']:.6f}")
print(f"Soporte X AND Y: {metricas_prueba['soporte_XY']:.6f}")
print(f"Confianza: {metricas_prueba['confianza']:.6f}")
print(f"Lift: {metricas_prueba['lift']:.6f}")
print(f"Factor de certeza: {metricas_prueba['factor_certeza']:.6f}")
print(f"¿Es dominante?: {validacion_particula['es_dominante']}")

if not validacion_particula["es_dominante"]:
    print("Criterios incumplidos:"f"{validacion_particula['criterios_incumplidos']}")

## 12. Generación de partículas aleatorias

Cada partícula inicial representa una regla candidata con entre una y cuatro condiciones.

Las variables se seleccionan aleatoriamente:

- Las continuas reciben un centro y un radio.
- Las categóricas reciben uno de sus valores válidos.
- Las variables no seleccionadas permanecen inactivas.

La selección se realiza sin repetir variables dentro de una misma regla.

In [ ]:
# ============================================================
# GENERAR PARTÍCULAS ALEATORIAS
# ============================================================

def crear_particula_aleatoria(
    generador,
    min_condiciones=MIN_CONDICIONES,
    max_condiciones=MAX_CONDICIONES,
    radio_minimo=RADIO_MINIMO,
    radio_maximo=RADIO_MAXIMO
):
    """Crea una partícula aleatoria válida."""

    total_variables = (
        len(columnas_continuas)
        + len(columnas_categoricas)
    )

    if min_condiciones < 1:
        raise ValueError(
            "'min_condiciones' debe ser al menos 1."
        )

    if max_condiciones < min_condiciones:
        raise ValueError(
            "'max_condiciones' no puede ser menor que "
            "'min_condiciones'."
        )

    if max_condiciones > total_variables:
        raise ValueError(
            "'max_condiciones' supera las variables disponibles."
        )

    if radio_minimo <= 0 or radio_maximo <= radio_minimo:
        raise ValueError(
            "Los límites del radio no son válidos."
        )

    particula = crear_particula_vacia()

    numero_condiciones = generador.integers(
        min_condiciones,
        max_condiciones + 1
    )

    variables_disponibles = (
        columnas_continuas
        + columnas_categoricas
    )

    variables_seleccionadas = generador.choice(
        variables_disponibles,
        size=numero_condiciones,
        replace=False
    )

    for variable in variables_seleccionadas:
        if variable in columnas_continuas:
            particula["continuas"][variable] = {
                "activa": 1,
                "centro": float(generador.uniform(0, 1)),
                "radio": float(
                    generador.uniform(
                        radio_minimo,
                        radio_maximo
                    )
                )
            }
        else:
            valor = generador.choice(
                valores_categoricos[variable]
            )

            if hasattr(valor, "item"):
                valor = valor.item()

            particula["categoricas"][variable] = {
                "activa": 1,
                "valor": valor
            }

    return particula

## 13. Velocidades iniciales

La velocidad indica cómo puede cambiar una partícula en la siguiente iteración.

Se manejarán tres tipos de movimiento:

- Activación o desactivación de variables.
- Movimiento del centro de una condición continua.
- Movimiento del radio de una condición continua.

Las variables categóricas no utilizan una velocidad numérica para sus valores, porque sus códigos representan categorías y no distancias.

In [ ]:
# ============================================================
# VELOCIDADES INICIALES
# ============================================================

RANGO_VELOCIDAD_ACTIVACION_INICIAL = 1.0
RANGO_VELOCIDAD_CENTRO_INICIAL = 0.05
RANGO_VELOCIDAD_RADIO_INICIAL = 0.02


def crear_velocidad_inicial(generador):
    """Crea las velocidades iniciales de una partícula."""

    velocidad = {
        "activacion": {},
        "continuas": {}
    }

    for variable in columnas_antecedente:
        velocidad["activacion"][variable] = float(
            generador.uniform(
                -RANGO_VELOCIDAD_ACTIVACION_INICIAL,
                RANGO_VELOCIDAD_ACTIVACION_INICIAL
            )
        )

    for variable in columnas_continuas:
        velocidad["continuas"][variable] = {
            "centro": float(
                generador.uniform(
                    -RANGO_VELOCIDAD_CENTRO_INICIAL,
                    RANGO_VELOCIDAD_CENTRO_INICIAL
                )
            ),
            "radio": float(
                generador.uniform(
                    -RANGO_VELOCIDAD_RADIO_INICIAL,
                    RANGO_VELOCIDAD_RADIO_INICIAL
                )
            )
        }

    return velocidad

## 14. Actualización de velocidades

PSO actualiza cada velocidad mediante tres componentes:

$$
v^{t+1}=
w v^t+
c_1r_1(pBest-x^t)+
c_2r_2(gBest-x^t)
$$

- \(w\): conserva parte del movimiento anterior.
- \(c_1\): atrae hacia la mejor posición personal.
- \(c_2\): atrae hacia la mejor posición global.

La misma idea se utiliza para activaciones, centros y radios.

In [ ]:
# ============================================================
# ACTUALIZAR VELOCIDADES
# ============================================================

W_INERCIA = 0.70
C1_COGNITIVO = 1.50
C2_SOCIAL = 1.50

VELOCIDAD_MAX_ACTIVACION = 4.0
VELOCIDAD_MAX_CENTRO = 0.20
VELOCIDAD_MAX_RADIO = 0.10


def obtener_estado_activacion(particula, variable):
    """Devuelve 1 si la variable está activa y 0 en otro caso."""

    if variable in columnas_continuas:
        return int(
            particula["continuas"][variable]["activa"]
        )

    if variable in columnas_categoricas:
        return int(
            particula["categoricas"][variable]["activa"]
        )

    raise ValueError(
        f"La variable '{variable}' no pertenece al espacio de búsqueda."
    )


def actualizar_velocidad_particula(
    particula_actual,
    velocidad_actual,
    pbest_particula,
    gbest_particula,
    generador,
    w=W_INERCIA,
    c1=C1_COGNITIVO,
    c2=C2_SOCIAL
):
    """Actualiza las velocidades de una partícula."""

    nueva_velocidad = copy.deepcopy(velocidad_actual)

    for variable in columnas_antecedente:
        estado_actual = obtener_estado_activacion(
            particula_actual,
            variable
        )
        estado_pbest = obtener_estado_activacion(
            pbest_particula,
            variable
        )
        estado_gbest = obtener_estado_activacion(
            gbest_particula,
            variable
        )

        velocidad = (
            w * velocidad_actual["activacion"][variable]
            + c1 * generador.random()
            * (estado_pbest - estado_actual)
            + c2 * generador.random()
            * (estado_gbest - estado_actual)
        )

        nueva_velocidad["activacion"][variable] = float(
            np.clip(
                velocidad,
                -VELOCIDAD_MAX_ACTIVACION,
                VELOCIDAD_MAX_ACTIVACION
            )
        )

    limites = {
        "centro": VELOCIDAD_MAX_CENTRO,
        "radio": VELOCIDAD_MAX_RADIO
    }

    for variable in columnas_continuas:
        for parametro, limite in limites.items():
            posicion_actual = particula_actual[
                "continuas"
            ][variable][parametro]

            posicion_pbest = pbest_particula[
                "continuas"
            ][variable][parametro]

            posicion_gbest = gbest_particula[
                "continuas"
            ][variable][parametro]

            velocidad_anterior = velocidad_actual[
                "continuas"
            ][variable][parametro]

            velocidad = (
                w * velocidad_anterior
                + c1 * generador.random()
                * (posicion_pbest - posicion_actual)
                + c2 * generador.random()
                * (posicion_gbest - posicion_actual)
            )

            nueva_velocidad["continuas"][variable][parametro] = float(
                np.clip(velocidad, -limite, limite)
            )

    return nueva_velocidad

## 15. Actualización de variables activas

Las variables del antecedente se actualizan mediante una versión binaria de PSO.

La velocidad se transforma en probabilidad mediante la función sigmoide:

$$
S(v)=\frac{1}{1+e^{-v}}
$$

Después se genera un número aleatorio:

$$
x=
\begin{cases}
1, & u<S(v) \\
0, & u\geq S(v)
\end{cases}
$$

Finalmente, la partícula se repara para mantener entre una y cuatro condiciones activas.

In [ ]:
# ============================================================
# ACTUALIZAR ACTIVACIONES
# ============================================================

def funcion_sigmoide(velocidad):
    """Convierte una velocidad en una probabilidad."""

    velocidad = np.clip(velocidad, -60, 60)
    return float(1 / (1 + np.exp(-velocidad)))


def establecer_estado_activacion(particula, variable, estado):
    """Activa o desactiva una variable."""

    if estado not in {0, 1}:
        raise ValueError("'estado' debe ser 0 o 1.")

    grupo = (
        "continuas"
        if variable in columnas_continuas
        else "categoricas"
    )

    particula[grupo][variable]["activa"] = int(estado)


def actualizar_activaciones_particula(
    particula_actual,
    velocidad_actualizada,
    generador,
    min_condiciones=MIN_CONDICIONES,
    max_condiciones=MAX_CONDICIONES
):
    """Actualiza las variables activas mediante sigmoide."""

    nueva_particula = copy.deepcopy(particula_actual)
    probabilidades = {}
    estados_aleatorios = {}

    for variable in columnas_antecedente:
        velocidad = velocidad_actualizada[
            "activacion"
        ][variable]

        probabilidad = funcion_sigmoide(velocidad)
        aleatorio = float(generador.random())
        estado = int(aleatorio < probabilidad)

        establecer_estado_activacion(
            nueva_particula,
            variable,
            estado
        )

        probabilidades[variable] = probabilidad
        estados_aleatorios[variable] = aleatorio

    variables_activas = [
        variable
        for variable in columnas_antecedente
        if obtener_estado_activacion(
            nueva_particula,
            variable
        ) == 1
    ]

    if len(variables_activas) < min_condiciones:
        variables_ordenadas = sorted(
            columnas_antecedente,
            key=probabilidades.get,
            reverse=True
        )

        for variable in variables_ordenadas[:min_condiciones]:
            establecer_estado_activacion(
                nueva_particula,
                variable,
                1
            )

    elif len(variables_activas) > max_condiciones:
        variables_conservar = set(
            sorted(
                variables_activas,
                key=probabilidades.get,
                reverse=True
            )[:max_condiciones]
        )

        for variable in variables_activas:
            if variable not in variables_conservar:
                establecer_estado_activacion(
                    nueva_particula,
                    variable,
                    0
                )

    filas_detalle = []

    for variable in columnas_antecedente:
        estado_final = obtener_estado_activacion(
            nueva_particula,
            variable
        )

        filas_detalle.append({
            "variable": variable,
            "velocidad": velocidad_actualizada[
                "activacion"
            ][variable],
            "probabilidad": probabilidades[variable],
            "aleatorio": estados_aleatorios[variable],
            "estado_final": estado_final
        })

    return (
        nueva_particula,
        pd.DataFrame(filas_detalle)
    )

## 16. Actualización de centros y radios

Las variables continuas tienen dos parámetros:

- \(c\): centro de la condición.
- \(R\): radio de activación.

El centro se mantiene dentro de \([0,1]\):

$$
c^{t+1}=
clip(c^t+v_c,0,1)
$$

El radio se mantiene dentro de los límites definidos:

$$
R^{t+1}=
clip(R^t+v_R,R_{min},R_{max})
$$

Esto evita que las partículas generen condiciones inválidas.

In [ ]:
# ============================================================
# ACTUALIZAR CENTROS Y RADIOS
# ============================================================

def actualizar_posiciones_continuas(
    particula_actual,
    velocidad_actualizada,
    centro_minimo=LIMITE_CENTRO_INFERIOR,
    centro_maximo=LIMITE_CENTRO_SUPERIOR,
    radio_minimo=RADIO_MINIMO,
    radio_maximo=RADIO_MAXIMO
):
    """Actualiza centros y radios de las variables continuas."""

    if centro_minimo >= centro_maximo:
        raise ValueError(
            "Los límites del centro no son válidos."
        )

    if radio_minimo <= 0 or radio_minimo >= radio_maximo:
        raise ValueError(
            "Los límites del radio no son válidos."
        )

    nueva_particula = copy.deepcopy(particula_actual)
    filas_detalle = []

    for variable in columnas_continuas:
        datos_actuales = particula_actual[
            "continuas"
        ][variable]

        velocidades = velocidad_actualizada[
            "continuas"
        ][variable]

        centro_anterior = datos_actuales["centro"]
        radio_anterior = datos_actuales["radio"]

        centro_propuesto = (
            centro_anterior
            + velocidades["centro"]
        )

        radio_propuesto = (
            radio_anterior
            + velocidades["radio"]
        )

        centro_nuevo = float(
            np.clip(
                centro_propuesto,
                centro_minimo,
                centro_maximo
            )
        )

        radio_nuevo = float(
            np.clip(
                radio_propuesto,
                radio_minimo,
                radio_maximo
            )
        )

        nueva_particula["continuas"][variable]["centro"] = (
            centro_nuevo
        )

        nueva_particula["continuas"][variable]["radio"] = (
            radio_nuevo
        )

        filas_detalle.append({
            "variable": variable,
            "centro_anterior": centro_anterior,
            "centro_nuevo": centro_nuevo,
            "radio_anterior": radio_anterior,
            "radio_nuevo": radio_nuevo
        })

    return (
        nueva_particula,
        pd.DataFrame(filas_detalle)
    )

## 17. Actualización de valores categóricos

Las categorías no se actualizan sumando velocidades, porque sus códigos son etiquetas y no distancias.

Por ejemplo, no tiene sentido interpretar:

$$
3-1=2
$$

como una distancia entre dos categorías de colesterol.

Por eso, cada valor categórico puede conservar el valor actual, copiar el valor de `pBest`, copiar el valor de `gBest` o explorar aleatoriamente otro valor válido.

In [ ]:
# ============================================================
# ACTUALIZAR VALORES CATEGÓRICOS
# ============================================================

def actualizar_valores_categoricos(
    particula_actual,
    pbest_particula,
    gbest_particula,
    generador,
    probabilidad_exploracion=0.10,
    w=W_INERCIA,
    c1=C1_COGNITIVO,
    c2=C2_SOCIAL
):
    """Actualiza los valores de las variables categóricas."""

    if not 0 <= probabilidad_exploracion <= 1:
        raise ValueError(
            "'probabilidad_exploracion' debe estar entre 0 y 1."
        )

    nueva_particula = copy.deepcopy(particula_actual)
    filas_detalle = []

    fuentes = np.array([
        "actual",
        "pBest",
        "gBest"
    ])

    for variable in columnas_categoricas:
        dominio = valores_categoricos[variable]

        valor_actual = particula_actual[
            "categoricas"
        ][variable]["valor"]

        valor_pbest = pbest_particula[
            "categoricas"
        ][variable]["valor"]

        valor_gbest = gbest_particula[
            "categoricas"
        ][variable]["valor"]

        if generador.random() < probabilidad_exploracion:
            valor_nuevo = generador.choice(dominio)
            fuente = "exploracion"
        else:
            pesos = np.array([
                w,
                c1 * generador.random(),
                c2 * generador.random()
            ])

            probabilidades = pesos / pesos.sum()
            fuente = generador.choice(
                fuentes,
                p=probabilidades
            )

            valores = {
                "actual": valor_actual,
                "pBest": valor_pbest,
                "gBest": valor_gbest
            }

            valor_nuevo = valores[fuente]

        if hasattr(valor_nuevo, "item"):
            valor_nuevo = valor_nuevo.item()

        nueva_particula[
            "categoricas"
        ][variable]["valor"] = valor_nuevo

        filas_detalle.append({
            "variable": variable,
            "valor_actual": valor_actual,
            "valor_pBest": valor_pbest,
            "valor_gBest": valor_gbest,
            "fuente": fuente,
            "valor_nuevo": valor_nuevo
        })

    return (
        nueva_particula,
        pd.DataFrame(filas_detalle)
    )

## 18. Iteración completa de PSO

En cada iteración, cada partícula realiza este proceso:

1. Actualiza su velocidad.
2. Actualiza las variables activas.
3. Actualiza centros y radios.
4. Actualiza valores categóricos.
5. Evalúa la nueva regla.
6. Actualiza su `pBest` si encontró una solución mejor.

Al finalizar todas las partículas, se actualiza el `gBest` del enjambre.

In [ ]:
# ============================================================
# EJECUTAR UNA ITERACIÓN DE PSO
# ============================================================

def ejecutar_iteracion_pso_dominante(
    enjambre,
    velocidades,
    pbest_particulas,
    pbest_fitness,
    pbest_resultados,
    gbest_particula,
    gbest_fitness,
    gbest_resultado,
    dataframe,
    objetivo,
    valor_consecuente,
    metodo_similitud,
    soporte_minimo,
    generador,
    probabilidad_exploracion=0.10
):
    """Ejecuta una iteración completa de PSO."""

    numero_particulas = len(enjambre)

    if not (
        len(velocidades)
        == len(pbest_particulas)
        == len(pbest_fitness)
        == len(pbest_resultados)
        == numero_particulas
    ):
        raise ValueError(
            "Las estructuras del enjambre no tienen el mismo tamaño."
        )

    gbest_inicio = copy.deepcopy(gbest_particula)
    gbest_fitness_inicio = float(gbest_fitness)

    nuevo_enjambre = []
    nuevas_velocidades = []
    nuevos_resultados = []

    nuevos_pbest_particulas = [
        copy.deepcopy(particula)
        for particula in pbest_particulas
    ]

    nuevos_pbest_fitness = np.array(
        pbest_fitness,
        dtype=float,
        copy=True
    )

    nuevos_pbest_resultados = [
        copy.deepcopy(resultado)
        for resultado in pbest_resultados
    ]

    filas_iteracion = []

    for indice in range(numero_particulas):
        particula_actual = enjambre[indice]

        velocidad_nueva = actualizar_velocidad_particula(
            particula_actual=particula_actual,
            velocidad_actual=velocidades[indice],
            pbest_particula=pbest_particulas[indice],
            gbest_particula=gbest_inicio,
            generador=generador
        )

        particula_actualizada, _ = (
            actualizar_activaciones_particula(
                particula_actual=particula_actual,
                velocidad_actualizada=velocidad_nueva,
                generador=generador
            )
        )

        particula_actualizada, _ = (
            actualizar_posiciones_continuas(
                particula_actual=particula_actualizada,
                velocidad_actualizada=velocidad_nueva
            )
        )

        particula_nueva, _ = (
            actualizar_valores_categoricos(
                particula_actual=particula_actualizada,
                pbest_particula=pbest_particulas[indice],
                gbest_particula=gbest_inicio,
                generador=generador,
                probabilidad_exploracion=(
                    probabilidad_exploracion
                )
            )
        )

        resultado_nuevo = evaluar_particula_dominante(
            particula=particula_nueva,
            dataframe=dataframe,
            objetivo=objetivo,
            valor_consecuente=valor_consecuente,
            metodo_similitud=metodo_similitud,
            soporte_minimo=soporte_minimo,
            max_condiciones=MAX_CONDICIONES
        )

        fitness_nuevo = resultado_nuevo["fitness"]
        mejora_pbest = (
            fitness_nuevo > nuevos_pbest_fitness[indice]
        )

        if mejora_pbest:
            nuevos_pbest_particulas[indice] = (
                copy.deepcopy(particula_nueva)
            )
            nuevos_pbest_fitness[indice] = fitness_nuevo
            nuevos_pbest_resultados[indice] = (
                copy.deepcopy(resultado_nuevo)
            )

        nuevo_enjambre.append(particula_nueva)
        nuevas_velocidades.append(velocidad_nueva)
        nuevos_resultados.append(resultado_nuevo)

        metricas = resultado_nuevo["metricas"]

        filas_iteracion.append({
            "particula": indice + 1,
            "fitness_pBest_anterior": pbest_fitness[indice],
            "fitness_nuevo": fitness_nuevo,
            "mejora_pBest": mejora_pbest,
            "fitness_pBest_nuevo": (
                nuevos_pbest_fitness[indice]
            ),
            "numero_condiciones": (
                metricas["numero_condiciones"]
            ),
            "soporte_XY": metricas["soporte_XY"],
            "confianza": metricas["confianza"],
            "lift": metricas["lift"],
            "factor_certeza": (
                metricas["factor_certeza"]
            )
        })

    indice_gbest = int(
        np.argmax(nuevos_pbest_fitness)
    )

    nuevo_gbest_particula = copy.deepcopy(
        nuevos_pbest_particulas[indice_gbest]
    )

    nuevo_gbest_fitness = float(
        nuevos_pbest_fitness[indice_gbest]
    )

    nuevo_gbest_resultado = copy.deepcopy(
        nuevos_pbest_resultados[indice_gbest]
    )

    tabla_iteracion = pd.DataFrame(filas_iteracion)

    resumen_iteracion = {
        "gbest_fitness_anterior": gbest_fitness_inicio,
        "gbest_fitness_nuevo": nuevo_gbest_fitness,
        "gbest_mejoro": (
            nuevo_gbest_fitness > gbest_fitness_inicio
        ),
        "indice_gbest": indice_gbest,
        "cantidad_pbest_actualizados": int(
            tabla_iteracion["mejora_pBest"].sum()
        )
    }

    return {
        "enjambre": nuevo_enjambre,
        "velocidades": nuevas_velocidades,
        "resultados_actuales": nuevos_resultados,
        "pbest_particulas": nuevos_pbest_particulas,
        "pbest_fitness": nuevos_pbest_fitness,
        "pbest_resultados": nuevos_pbest_resultados,
        "gbest_particula": nuevo_gbest_particula,
        "gbest_fitness": nuevo_gbest_fitness,
        "gbest_resultado": nuevo_gbest_resultado,
        "tabla_iteracion": tabla_iteracion,
        "resumen_iteracion": resumen_iteracion
    }

## 19. Ejecución completa del PSO

Esta función ejecuta todo el proceso:

1. Genera el enjambre inicial.
2. Crea las velocidades.
3. Evalúa las partículas.
4. Inicializa `pBest` y `gBest`.
5. Ejecuta las iteraciones.
6. Guarda el historial de convergencia.

En esta etapa todavía no se ejecuta ninguna búsqueda. Solo se define la función principal.

In [ ]:
# ============================================================
# EJECUTAR PSO PARA REGLAS DOMINANTES
# ============================================================

def ejecutar_pso_reglas_dominantes(
    dataframe,
    objetivo,
    valor_consecuente,
    numero_particulas=20,
    numero_iteraciones=30,
    metodo_similitud="triangular",
    soporte_minimo=0.005,
    probabilidad_exploracion=0.10,
    semilla=2030
):
    """Ejecuta el PSO completo para una configuración."""

    if numero_particulas < 2:
        raise ValueError(
            "'numero_particulas' debe ser al menos 2."
        )

    if numero_iteraciones < 1:
        raise ValueError(
            "'numero_iteraciones' debe ser al menos 1."
        )

    if valor_consecuente not in {0, 1}:
        raise ValueError(
            "'valor_consecuente' debe ser 0 o 1."
        )

    if metodo_similitud not in {
        "triangular",
        "gaussiana"
    }:
        raise ValueError(
            "Método de similaridad no válido."
        )

    generador = np.random.default_rng(semilla)

    enjambre = [
        crear_particula_aleatoria(generador)
        for _ in range(numero_particulas)
    ]

    velocidades = [
        crear_velocidad_inicial(generador)
        for _ in range(numero_particulas)
    ]

    resultados_actuales = [
        evaluar_particula_dominante(
            particula=particula,
            dataframe=dataframe,
            objetivo=objetivo,
            valor_consecuente=valor_consecuente,
            metodo_similitud=metodo_similitud,
            soporte_minimo=soporte_minimo,
            max_condiciones=MAX_CONDICIONES
        )
        for particula in enjambre
    ]

    pbest_particulas = copy.deepcopy(enjambre)
    pbest_resultados = copy.deepcopy(resultados_actuales)

    pbest_fitness = np.array([
        resultado["fitness"]
        for resultado in resultados_actuales
    ])

    indice_gbest = int(np.argmax(pbest_fitness))

    gbest_particula = copy.deepcopy(
        pbest_particulas[indice_gbest]
    )

    gbest_fitness = float(
        pbest_fitness[indice_gbest]
    )

    gbest_resultado = copy.deepcopy(
        pbest_resultados[indice_gbest]
    )

    historial_filas = []
    historial_gbest = [
        copy.deepcopy(gbest_particula)
    ]

    fitness_actuales = np.array([
        resultado["fitness"]
        for resultado in resultados_actuales
    ])

    def registrar_historial(iteracion, actualizados):
        metricas = gbest_resultado["metricas"]

        historial_filas.append({
            "iteracion": iteracion,
            "gbest_fitness": gbest_fitness,
            "fitness_promedio_actual": (
                fitness_actuales.mean()
            ),
            "fitness_maximo_actual": (
                fitness_actuales.max()
            ),
            "particulas_fitness_positivo": int(
                np.sum(fitness_actuales > 0)
            ),
            "pbest_actualizados": actualizados,
            "numero_condiciones_gbest": (
                metricas["numero_condiciones"]
            ),
            "soporte_XY_gbest": metricas["soporte_XY"],
            "confianza_gbest": metricas["confianza"],
            "lift_gbest": metricas["lift"],
            "factor_certeza_gbest": (
                metricas["factor_certeza"]
            )
        })

    registrar_historial(0, 0)

    for iteracion in range(1, numero_iteraciones + 1):
        resultado_iteracion = (
            ejecutar_iteracion_pso_dominante(
                enjambre=enjambre,
                velocidades=velocidades,
                pbest_particulas=pbest_particulas,
                pbest_fitness=pbest_fitness,
                pbest_resultados=pbest_resultados,
                gbest_particula=gbest_particula,
                gbest_fitness=gbest_fitness,
                gbest_resultado=gbest_resultado,
                dataframe=dataframe,
                objetivo=objetivo,
                valor_consecuente=valor_consecuente,
                metodo_similitud=metodo_similitud,
                soporte_minimo=soporte_minimo,
                generador=generador,
                probabilidad_exploracion=(
                    probabilidad_exploracion
                )
            )
        )

        enjambre = resultado_iteracion["enjambre"]
        velocidades = resultado_iteracion["velocidades"]
        resultados_actuales = resultado_iteracion[
            "resultados_actuales"
        ]
        pbest_particulas = resultado_iteracion[
            "pbest_particulas"
        ]
        pbest_fitness = resultado_iteracion[
            "pbest_fitness"
        ]
        pbest_resultados = resultado_iteracion[
            "pbest_resultados"
        ]
        gbest_particula = resultado_iteracion[
            "gbest_particula"
        ]
        gbest_fitness = resultado_iteracion[
            "gbest_fitness"
        ]
        gbest_resultado = resultado_iteracion[
            "gbest_resultado"
        ]

        fitness_actuales = np.array([
            resultado["fitness"]
            for resultado in resultados_actuales
        ])

        actualizados = resultado_iteracion[
            "resumen_iteracion"
        ]["cantidad_pbest_actualizados"]

        registrar_historial(iteracion, actualizados)

        historial_gbest.append(
            copy.deepcopy(gbest_particula)
        )

    return {
        "enjambre_final": enjambre,
        "velocidades_finales": velocidades,
        "resultados_actuales": resultados_actuales,
        "pbest_particulas": pbest_particulas,
        "pbest_fitness": pbest_fitness,
        "pbest_resultados": pbest_resultados,
        "gbest_particula": gbest_particula,
        "gbest_fitness": gbest_fitness,
        "gbest_resultado": gbest_resultado,
        "historial_convergencia": pd.DataFrame(
            historial_filas
        ),
        "historial_gbest": historial_gbest,
        "configuracion": {
            "numero_particulas": numero_particulas,
            "numero_iteraciones": numero_iteraciones,
            "valor_consecuente": valor_consecuente,
            "metodo_similitud": metodo_similitud,
            "soporte_minimo": soporte_minimo,
            "probabilidad_exploracion": (
                probabilidad_exploracion
            ),
            "semilla": semilla
        }
    }

## 20. Prueba individual opcional

Esta prueba permite verificar el flujo con un enjambre pequeño antes de ejecutar el experimento completo.

La prueba está desactivada por defecto y no modifica las estructuras utilizadas por el experimento multisemilla.

In [ ]:
# ============================================================
# PRUEBA INDIVIDUAL OPCIONAL
# ============================================================

EJECUTAR_PRUEBA_INDIVIDUAL = False

if EJECUTAR_PRUEBA_INDIVIDUAL:
    resultado_prueba_individual = ejecutar_pso_reglas_dominantes(
        dataframe=X_datos,
        objetivo=y_objetivo,
        valor_consecuente=1,
        numero_particulas=20,
        numero_iteraciones=30,
        metodo_similitud="triangular",
        soporte_minimo=0.005,
        probabilidad_exploracion=0.10,
        semilla=2030
    )

    gbest_prueba = resultado_prueba_individual["gbest_resultado"]
    metricas_prueba = gbest_prueba["metricas"]

    validacion_prueba = verificar_regla_dominante(
        resultado_regla=gbest_prueba,
        soporte_minimo=0.005,
        confianza_minima=0.60,
        lift_minimo=1.00,
        factor_certeza_minimo=0.10,
        min_condiciones=MIN_CONDICIONES,
        max_condiciones=MAX_CONDICIONES
    )

    print("Regla evaluada:")
    for condicion in gbest_prueba["condiciones"]:
        print(f"  {condicion}")

    print(f"\nFitness: {resultado_prueba_individual['gbest_fitness']:.8f}")
    print(f"Soporte X: {metricas_prueba['soporte_X']:.6f}")
    print(f"Soporte X AND Y: {metricas_prueba['soporte_XY']:.6f}")
    print(f"Confianza: {metricas_prueba['confianza']:.6f}")
    print(f"Lift: {metricas_prueba['lift']:.6f}")
    print(f"Factor de certeza: {metricas_prueba['factor_certeza']:.6f}")
    print(f"Es dominante: {validacion_prueba['es_dominante']}")

    display(
        resultado_prueba_individual["historial_convergencia"]
        .tail()
        .round(6)
    )
else:
    resultado_prueba_individual = None
    print("Prueba individual omitida; se ejecutar? solamente el experimento completo.")


## 21. Conversión a unidades originales

Las partículas trabajan con valores normalizados, pero las reglas deben poder interpretarse en las unidades originales del dataset.

Para un centro normalizado:

$$
x_{original}=
L+x_{normalizado}(U-L)
$$

Para un radio:

$$
R_{original}=
R_{normalizado}(U-L)
$$

donde \(L\) y \(U\) son los límites robustos guardados para cada variable.

In [ ]:
# ============================================================
# CONVERTIR VALORES A UNIDADES ORIGINALES
# ============================================================

def convertir_normalizado_a_original(
    valor_normalizado,
    columna,
    parametros
):
    """Convierte un valor normalizado a su escala original."""

    if columna not in parametros:
        raise ValueError(
            f"No existen parámetros para '{columna}'."
        )

    if not 0 <= valor_normalizado <= 1:
        raise ValueError(
            "'valor_normalizado' debe estar entre 0 y 1."
        )

    limite_inferior = parametros[columna][
        "limite_inferior"
    ]

    amplitud = parametros[columna]["amplitud"]

    return (
        limite_inferior
        + valor_normalizado * amplitud
    )


def convertir_radio_a_original(
    radio_normalizado,
    columna,
    parametros
):
    """Convierte un radio normalizado a su escala original."""

    if columna not in parametros:
        raise ValueError(
            f"No existen parámetros para '{columna}'."
        )

    if radio_normalizado <= 0:
        raise ValueError(
            "'radio_normalizado' debe ser positivo."
        )

    amplitud = parametros[columna]["amplitud"]

    return radio_normalizado * amplitud

## 22. Interpretación de la mejor regla

La mejor partícula contiene centros, radios y códigos categóricos. Esta función traduce esos valores a una descripción comprensible.

Por ejemplo:

```text
presion_sistolica = 0.52

In [ ]:
# ============================================================
# TRADUCIR LA MEJOR REGLA
# ============================================================

def obtener_etiqueta_categoria(
    variable,
    valor,
    mapas
):
    """Convierte un código categórico en una etiqueta."""

    if isinstance(valor, (float, np.floating)):
        if valor.is_integer():
            valor = int(valor)

    return mapas.get(variable, {}).get(
        valor,
        str(valor)
    )


def traducir_regla_dominante(
    resultado_pso,
    parametros_normalizacion,
    mapas,
    columna_consecuente
):
    """Traduce la mejor regla a unidades originales."""

    resultado = resultado_pso["gbest_resultado"]
    condiciones = resultado["condiciones"]
    valor_consecuente = resultado[
        "metricas"
    ]["valor_consecuente"]

    filas = []
    textos = []

    for condicion in condiciones:
        variable = condicion["variable"]
        tipo = condicion["tipo"]

        if tipo == "continua":
            centro_norm = float(condicion["centro"])
            radio_norm = float(condicion["radio"])

            centro = convertir_normalizado_a_original(
                centro_norm,
                variable,
                parametros_normalizacion
            )

            radio = convertir_radio_a_original(
                radio_norm,
                variable,
                parametros_normalizacion
            )

            limite_inf_norm = max(
                0.0,
                centro_norm - radio_norm
            )

            limite_sup_norm = min(
                1.0,
                centro_norm + radio_norm
            )

            limite_inf = convertir_normalizado_a_original(
                limite_inf_norm,
                variable,
                parametros_normalizacion
            )

            limite_sup = convertir_normalizado_a_original(
                limite_sup_norm,
                variable,
                parametros_normalizacion
            )

            texto = (
                f"{variable} cercana a {centro:.2f} "
                f"con radio ±{radio:.2f}"
            )

            filas.append({
                "variable": variable,
                "tipo": "continua",
                "centro_normalizado": centro_norm,
                "radio_normalizado": radio_norm,
                "centro_original": centro,
                "radio_original": radio,
                "limite_original_inferior": limite_inf,
                "limite_original_superior": limite_sup,
                "valor_categoria": np.nan,
                "etiqueta_categoria": None,
                "condicion_interpretada": texto
            })

        else:
            valor = condicion["valor"]

            etiqueta = obtener_etiqueta_categoria(
                variable,
                valor,
                mapas
            )

            texto = f"{variable} = {etiqueta}"

            filas.append({
                "variable": variable,
                "tipo": "categórica",
                "centro_normalizado": np.nan,
                "radio_normalizado": np.nan,
                "centro_original": np.nan,
                "radio_original": np.nan,
                "limite_original_inferior": np.nan,
                "limite_original_superior": np.nan,
                "valor_categoria": valor,
                "etiqueta_categoria": etiqueta,
                "condicion_interpretada": texto
            })

        textos.append(texto)

    etiqueta_consecuente = obtener_etiqueta_categoria(
        columna_consecuente,
        valor_consecuente,
        mapas
    )

    texto_consecuente = (
        f"{columna_consecuente} = "
        f"{etiqueta_consecuente}"
    )

    return {
        "texto_antecedente": " Y ".join(textos),
        "texto_consecuente": texto_consecuente,
        "texto_regla": (
            f"{' Y '.join(textos)} "
            f"=> {texto_consecuente}"
        ),
        "tabla_condiciones": pd.DataFrame(filas)
    }


# ============================================================
# MOSTRAR LA REGLA INTERPRETADA

if EJECUTAR_PRUEBA_INDIVIDUAL:
    regla_traducida = traducir_regla_dominante(
        resultado_pso={
            "gbest_resultado": resultado_prueba_individual["gbest_resultado"]
        },
        parametros_normalizacion=parametros_normalizacion_robusta,
        mapas=mapas_categorias,
        columna_consecuente=columna_objetivo
    )

    print("Regla encontrada en la prueba individual:")
    print(regla_traducida["texto_regla"])
    display(regla_traducida["tabla_condiciones"].round(4))
else:
    regla_traducida = None


## 23. Configuración del experimento comparativo

Se compararán tres tamaños de enjambre: 50, 100 y 500 partículas.

Para cada tamaño se ejecutarán cuatro configuraciones:

1. Enfermedad cardiovascular = sí con similaridad triangular.
2. Enfermedad cardiovascular = sí con similaridad gaussiana.
3. Enfermedad cardiovascular = no con similaridad triangular.
4. Enfermedad cardiovascular = no con similaridad gaussiana.

Cada configuración se ejecutará con las mismas semillas para facilitar la comparación entre tamaños de enjambre.

In [ ]:
# ============================================================
# CONFIGURACION DEL EXPERIMENTO COMPARATIVO
# ============================================================

NOMBRE_EXPERIMENTO = "experimento_pso_reglas_dominantes"

SEMILLAS_EXPERIMENTO = [55, 13, 33]
NUMERO_ITERACIONES_EXPERIMENTO = 50

TAMANOS_ENJAMBRE = [
    {
        "nombre": "enjambre_50",
        "numero_particulas": 50,
        "numero_iteraciones": NUMERO_ITERACIONES_EXPERIMENTO
    },
    {
        "nombre": "enjambre_100",
        "numero_particulas": 100,
        "numero_iteraciones": NUMERO_ITERACIONES_EXPERIMENTO
    },
    {
        "nombre": "enjambre_500",
        "numero_particulas": 500,
        "numero_iteraciones": NUMERO_ITERACIONES_EXPERIMENTO
    }
]

SOPORTE_MINIMO_DOMINANTE = 0.01
CONFIANZA_MINIMA_DOMINANTE = 0.60
LIFT_MINIMO_DOMINANTE = 1.00
CF_MINIMO_DOMINANTE = 0.10
PROBABILIDAD_EXPLORACION_EXPERIMENTO = 0.10

configuraciones_experimento = [
    {
        "nombre": "cardio_si_triangular",
        "valor_consecuente": 1,
        "metodo_similitud": "triangular"
    },
    {
        "nombre": "cardio_si_gaussiana",
        "valor_consecuente": 1,
        "metodo_similitud": "gaussiana"
    },
    {
        "nombre": "cardio_no_triangular",
        "valor_consecuente": 0,
        "metodo_similitud": "triangular"
    },
    {
        "nombre": "cardio_no_gaussiana",
        "valor_consecuente": 0,
        "metodo_similitud": "gaussiana"
    }
]

total_ejecuciones = (
    len(TAMANOS_ENJAMBRE)
    * len(configuraciones_experimento)
    * len(SEMILLAS_EXPERIMENTO)
)

print(f"Experimento: {NOMBRE_EXPERIMENTO}")
print(
    "Tama?os de enjambre: "
    f"{[tamano['numero_particulas'] for tamano in TAMANOS_ENJAMBRE]}"
)
print(f"Iteraciones por tama?o: {NUMERO_ITERACIONES_EXPERIMENTO}")
print(f"Configuraciones: {len(configuraciones_experimento)}")
print(f"Semillas por configuraci?n: {len(SEMILLAS_EXPERIMENTO)}")
print(f"Total de ejecuciones: {total_ejecuciones}")


## 24. Ejecución comparativa multisemilla

Cada combinación de tamaño de enjambre, configuración y semilla se ejecutará de forma independiente.

Se conservarán:

- las reglas pBest que cumplen los criterios de dominancia;
- la mejor regla gBest de cada ejecución;
- las métricas de convergencia;
- el tamaño del enjambre utilizado.

El experimento completo tendrá 3 tamaños x 4 configuraciones x 3 semillas = 36 ejecuciones.

In [ ]:
# ============================================================
# EJECUTAR EXPERIMENTO COMPARATIVO MULTISEMILLA
# ============================================================

def compactar_resultado_regla(resultado):
    """Elimina el vector de activacion para ahorrar memoria."""

    metricas = {
        clave: copy.deepcopy(valor)
        for clave, valor in resultado["metricas"].items()
        if clave != "activacion_antecedente"
    }

    return {
        "condiciones": copy.deepcopy(resultado["condiciones"]),
        "metricas": metricas,
        "componentes_fitness": copy.deepcopy(resultado["componentes_fitness"]),
        "fitness": float(resultado["fitness"])
    }


reglas_dominantes_multisemilla = []
filas_resumen_ejecuciones = []
numero_ejecucion = 0

for tamano in TAMANOS_ENJAMBRE:
    for configuracion in configuraciones_experimento:
        for semilla in SEMILLAS_EXPERIMENTO:
            numero_ejecucion += 1

            nombre_tamano = tamano["nombre"]
            numero_particulas = tamano["numero_particulas"]
            numero_iteraciones = tamano["numero_iteraciones"]
            nombre_configuracion = configuracion["nombre"]
            configuracion_completa = (
                f"{nombre_tamano}__{nombre_configuracion}"
            )
            valor_consecuente = configuracion["valor_consecuente"]
            metodo_similitud = configuracion["metodo_similitud"]

            resultado_pso = ejecutar_pso_reglas_dominantes(
                dataframe=X_datos,
                objetivo=y_objetivo,
                valor_consecuente=valor_consecuente,
                numero_particulas=numero_particulas,
                numero_iteraciones=numero_iteraciones,
                metodo_similitud=metodo_similitud,
                soporte_minimo=SOPORTE_MINIMO_DOMINANTE,
                probabilidad_exploracion=(
                    PROBABILIDAD_EXPLORACION_EXPERIMENTO
                ),
                semilla=semilla
            )

            cantidad_dominantes = 0

            for indice, resultado_regla in enumerate(
                resultado_pso["pbest_resultados"],
                start=1
            ):
                validacion = verificar_regla_dominante(
                    resultado_regla=resultado_regla,
                    soporte_minimo=SOPORTE_MINIMO_DOMINANTE,
                    confianza_minima=CONFIANZA_MINIMA_DOMINANTE,
                    lift_minimo=LIFT_MINIMO_DOMINANTE,
                    factor_certeza_minimo=CF_MINIMO_DOMINANTE,
                    min_condiciones=MIN_CONDICIONES,
                    max_condiciones=MAX_CONDICIONES
                )

                if not validacion["es_dominante"]:
                    continue

                cantidad_dominantes += 1

                reglas_dominantes_multisemilla.append({
                    "configuracion": configuracion_completa,
                    "configuracion_base": nombre_configuracion,
                    "tamano_enjambre": nombre_tamano,
                    "numero_particulas": numero_particulas,
                    "numero_iteraciones": numero_iteraciones,
                    "semilla": semilla,
                    "particula": indice,
                    "valor_consecuente": valor_consecuente,
                    "metodo_similitud": metodo_similitud,
                    "resultado": compactar_resultado_regla(
                        resultado_regla
                    )
                })

            gbest = resultado_pso["gbest_resultado"]
            metricas = gbest["metricas"]

            traduccion = traducir_regla_dominante(
                resultado_pso={"gbest_resultado": gbest},
                parametros_normalizacion=(
                    parametros_normalizacion_robusta
                ),
                mapas=mapas_categorias,
                columna_consecuente=columna_objetivo
            )

            filas_resumen_ejecuciones.append({
                "tamano_enjambre": nombre_tamano,
                "numero_particulas": numero_particulas,
                "numero_iteraciones": numero_iteraciones,
                "configuracion": nombre_configuracion,
                "configuracion_completa": configuracion_completa,
                "semilla": semilla,
                "valor_consecuente": valor_consecuente,
                "metodo_similitud": metodo_similitud,
                "regla_gbest": traduccion["texto_regla"],
                "fitness_gbest": resultado_pso["gbest_fitness"],
                "soporte_XY_gbest": metricas["soporte_XY"],
                "confianza_gbest": metricas["confianza"],
                "lift_gbest": metricas["lift"],
                "factor_certeza_gbest": metricas["factor_certeza"],
                "numero_condiciones_gbest": (
                    metricas["numero_condiciones"]
                ),
                "reglas_dominantes_pbest": cantidad_dominantes
            })

            print(
                f"{numero_ejecucion}/{total_ejecuciones} | "
                f"{configuracion_completa} | semilla={semilla} | "
                f"dominantes={cantidad_dominantes} | "
                f"gBest={resultado_pso['gbest_fitness']:.6f}"
            )

            del resultado_pso
            gc.collect()

tabla_resumen_multisemilla = pd.DataFrame(
    filas_resumen_ejecuciones
)

resumen_tamanos_enjambre = (
    tabla_resumen_multisemilla
    .groupby("numero_particulas", as_index=False)
    .agg(
        tamano_enjambre=("tamano_enjambre", "first"),
        ejecuciones=("semilla", "count"),
        fitness_gbest_promedio=("fitness_gbest", "mean"),
        confianza_gbest_promedio=("confianza_gbest", "mean"),
        lift_gbest_promedio=("lift_gbest", "mean"),
        reglas_dominantes_pbest_promedio=(
            "reglas_dominantes_pbest",
            "mean"
        )
    )
    .sort_values("numero_particulas")
    .reset_index(drop=True)
)

print("\nResumen de ejecuciones:")
display(tabla_resumen_multisemilla.round(4))

print("\nComparaci?n por tama?o de enjambre:")
display(resumen_tamanos_enjambre.round(4))


In [ ]:

# ============================================================
# CONSOLIDAR REGLAS MULTISEMILLA
# ============================================================

def crear_firma_parametrica_regla(
    resultado_regla,
    metodo_similitud,
    decimales=4
):
    """Crea una firma comparable para una regla."""

    condiciones = sorted(
        resultado_regla["condiciones"],
        key=lambda condicion: condicion["variable"]
    )

    firma_condiciones = []

    for condicion in condiciones:
        variable = condicion["variable"]
        tipo = condicion["tipo"]

        if tipo == "continua":
            firma_condiciones.append((
                variable,
                tipo,
                round(float(condicion["centro"]), decimales),
                round(float(condicion["radio"]), decimales)
            ))

        elif tipo == "categorica":
            valor = condicion["valor"]
            valor = valor.item() if hasattr(valor, "item") else valor
            firma_condiciones.append((variable, tipo, valor))

    return (
        int(resultado_regla["metricas"]["valor_consecuente"]),
        metodo_similitud,
        tuple(firma_condiciones)
    )


def consolidar_reglas_multisemilla(
    reglas_encontradas,
    parametros_normalizacion,
    mapas,
    columna_consecuente,
    decimales_firma=4
):
    """Agrupa reglas con la misma firma paramétrica."""

    reglas_por_firma = {}

    for elemento in reglas_encontradas:
        resultado = elemento["resultado"]

        firma = crear_firma_parametrica_regla(
            resultado,
            elemento["metodo_similitud"],
            decimales_firma
        )

        if firma not in reglas_por_firma:
            reglas_por_firma[firma] = {
                "configuracion": elemento["configuracion"],
                "semilla": elemento["semilla"],
                "particula": elemento["particula"],
                "valor_consecuente": elemento["valor_consecuente"],
                "metodo_similitud": elemento["metodo_similitud"],
                "resultado": copy.deepcopy(resultado),
                "apariciones": 1,
                "semillas": {elemento["semilla"]},
                "configuraciones": {elemento["configuracion"]},
                "tamanos_enjambre": {elemento["numero_particulas"]}
            }
            continue

        guardada = reglas_por_firma[firma]
        guardada["apariciones"] += 1
        guardada["semillas"].add(elemento["semilla"])
        guardada["configuraciones"].add(elemento["configuracion"])
        guardada["tamanos_enjambre"].add(elemento["numero_particulas"])

        if resultado["fitness"] > guardada["resultado"]["fitness"]:
            guardada.update({
                "configuracion": elemento["configuracion"],
                "semilla": elemento["semilla"],
                "particula": elemento["particula"],
                "resultado": copy.deepcopy(resultado)
            })

    reglas_consolidadas = sorted(
        reglas_por_firma.values(),
        key=lambda elemento: elemento["resultado"]["fitness"],
        reverse=True
    )

    filas = []

    for posicion, elemento in enumerate(reglas_consolidadas, start=1):
        resultado = elemento["resultado"]
        metricas = resultado["metricas"]

        traduccion = traducir_regla_dominante(
            {"gbest_resultado": resultado},
            parametros_normalizacion,
            mapas,
            columna_consecuente
        )

        filas.append({
            "posicion": posicion,
            "regla": traduccion["texto_regla"],
            "consecuente": mapas[columna_consecuente][
                elemento["valor_consecuente"]
            ],
            "metodo_similitud": elemento["metodo_similitud"],
            "numero_condiciones": metricas["numero_condiciones"],
            "fitness": resultado["fitness"],
            "soporte_X": metricas["soporte_X"],
            "soporte_XY": metricas["soporte_XY"],
            "confianza": metricas["confianza"],
            "lift": metricas["lift"],
            "factor_certeza": metricas["factor_certeza"],
            "apariciones": elemento["apariciones"],
            "numero_semillas": len(elemento["semillas"]),
            "semillas": ", ".join(
                str(semilla) for semilla in sorted(elemento["semillas"])
            ),
            "configuraciones": ", ".join(
                sorted(elemento["configuraciones"])
            ),
            "numero_tamanos_enjambre": len(
                elemento["tamanos_enjambre"]
            ),
            "tamanos_enjambre": ", ".join(
                str(tamano)
                for tamano in sorted(elemento["tamanos_enjambre"])
            )
        })

    return reglas_consolidadas, pd.DataFrame(filas)


(
    reglas_dominantes_consolidadas,
    tabla_reglas_consolidadas
) = consolidar_reglas_multisemilla(
    reglas_dominantes_multisemilla,
    parametros_normalizacion_robusta,
    mapas_categorias,
    columna_objetivo
)

cantidad_antes = len(reglas_dominantes_multisemilla)
cantidad_despues = len(reglas_dominantes_consolidadas)

print(
    f"Reglas antes de consolidar: {cantidad_antes}"
    f"\nReglas después de consolidar: {cantidad_despues}"
    f"\nDuplicados paramétricos eliminados: {cantidad_antes - cantidad_despues}"
)

display(
    tabla_reglas_consolidadas.head(10).round({
        "fitness": 8,
        "soporte_X": 6,
        "soporte_XY": 6,
        "confianza": 6,
        "lift": 6,
        "factor_certeza": 6
    })
)

## 26. Redundancia con Jaccard difuso

La consolidación paramétrica detecta reglas con centros y radios prácticamente iguales. Sin embargo, dos reglas con parámetros diferentes todavía pueden activar casi a los mismos pacientes.

Jaccard difuso compara directamente los vectores de activación de dos antecedentes:

$$
J(A,B)=
\frac{
\sum_i \min\left(\mu_A(i),\mu_B(i)\right)
}{
\sum_i \max\left(\mu_A(i),\mu_B(i)\right)
}
$$

El numerador representa la intersección difusa: la activación compartida por ambas reglas. El denominador representa la unión difusa: toda la activación cubierta por al menos una de ellas.

Su resultado se encuentra entre 0 y 1:

- Un valor cercano a 0 indica poca coincidencia.
- Un valor cercano a 1 indica activaciones prácticamente iguales.
- Un valor igual a 1 representa vectores de activación idénticos.

Por ejemplo:

```text
Regla A: [1.0, 0.8, 0.0, 0.2]
Regla B: [0.9, 0.7, 0.0, 0.3]

```
La intersección es:
$$
0.9+0.7+0+0.2=1.8
$$La unión es:
$$
1.0+0.8+0+0.3=2.1
$$Por tanto:
$$
J(A,B)=\frac{1.8}{2.1}=0.8571
$$Como supera el umbral de 0.85, ambas reglas se consideran miembros de la misma familia.
Para evitar comparaciones entre reglas conceptualmente diferentes, solamente se comparan reglas que tengan:

1.   El mismo consecuente.

2.   El mismo método de similaridad.
3.   Las mismas variables en el antecedente.

In [ ]:
# ============================================================
# FILTRAR REDUNDANCIA CON JACCARD DIFUSO
# ============================================================

def jaccard_difuso(activacion_a, activacion_b):
    """Calcula Jaccard entre dos vectores de activación difusa."""

    activacion_a = np.asarray(activacion_a, dtype=float)
    activacion_b = np.asarray(activacion_b, dtype=float)

    if len(activacion_a) != len(activacion_b):
        raise ValueError("Las activaciones deben tener la misma longitud.")

    interseccion = np.minimum(activacion_a, activacion_b).sum()
    union = np.maximum(activacion_a, activacion_b).sum()

    return float(interseccion / union) if union > 0 else 0.0


def obtener_firma_variables(resultado_regla):
    """Obtiene las variables que forman el antecedente."""

    return tuple(sorted(
        condicion["variable"]
        for condicion in resultado_regla["condiciones"]
    ))


def filtrar_reglas_por_jaccard(
    reglas_consolidadas,
    dataframe,
    parametros_normalizacion,
    mapas,
    columna_consecuente,
    umbral_jaccard=0.85
):
    """Agrupa reglas con activaciones difusas semejantes."""

    # 1. Crear grupos comparables
    grupos = {}

    for elemento in reglas_consolidadas:
        resultado = elemento["resultado"]

        clave_grupo = (
            elemento["valor_consecuente"],
            elemento["metodo_similitud"],
            obtener_firma_variables(resultado)
        )

        grupos.setdefault(clave_grupo, []).append(elemento)

    reglas_no_redundantes = []
    reglas_descartadas = []
    comparaciones = 0

    # 2. Procesar cada grupo
    for reglas_grupo in grupos.values():
        reglas_grupo = sorted(
            reglas_grupo,
            key=lambda elemento: elemento["resultado"]["fitness"],
            reverse=True
        )

        representantes = []
        activaciones_representantes = []

        for candidata in reglas_grupo:
            resultado = candidata["resultado"]

            activacion = calcular_activacion_antecedente(
                dataframe=dataframe,
                condiciones=resultado["condiciones"],
                metodo_similitud=candidata["metodo_similitud"],
                devolver_detalle=False
            ).astype(np.float32)

            redundante = False
            mayor_jaccard = 0.0
            indice_representante = None

            # 3. Comparar con las familias existentes
            for indice, activacion_representante in enumerate(
                activaciones_representantes
            ):
                comparaciones += 1

                similitud = jaccard_difuso(
                    activacion,
                    activacion_representante
                )

                if similitud > mayor_jaccard:
                    mayor_jaccard = similitud
                    indice_representante = indice

                if similitud >= umbral_jaccard:
                    redundante = True
                    break

            # 4. Incorporar a una familia existente
            if redundante:
                representante = representantes[indice_representante]

                representante["miembros_familia_jaccard"] += candidata.get(
                    "apariciones", 1
                )
                representante["semillas"].update(
                    candidata.get("semillas", {candidata["semilla"]})
                )
                representante["configuraciones"].update(
                    candidata.get(
                        "configuraciones",
                        {candidata["configuracion"]}
                    )
                )
                representante.setdefault(
                    "tamanos_enjambre",
                    set()
                ).update(
                    candidata.get(
                        "tamanos_enjambre",
                        set()
                    )
                )
                representante["jaccard_maximo_familia"] = max(
                    representante["jaccard_maximo_familia"],
                    mayor_jaccard
                )

                reglas_descartadas.append({
                    "fitness": resultado["fitness"],
                    "jaccard": mayor_jaccard,
                    "variables": ", ".join(
                        obtener_firma_variables(resultado)
                    )
                })

            # 5. Crear una nueva familia
            else:
                nueva_representante = copy.deepcopy(candidata)
                nueva_representante["miembros_familia_jaccard"] = (
                    candidata.get("apariciones", 1)
                )
                nueva_representante["jaccard_maximo_familia"] = 0.0

                representantes.append(nueva_representante)
                activaciones_representantes.append(activacion)

        reglas_no_redundantes.extend(representantes)

    # 6. Ordenar las reglas representativas
    reglas_no_redundantes.sort(
        key=lambda elemento: elemento["resultado"]["fitness"],
        reverse=True
    )

    # 7. Crear la tabla final
    filas = []

    for posicion, elemento in enumerate(reglas_no_redundantes, start=1):
        resultado = elemento["resultado"]
        metricas = resultado["metricas"]

        traduccion = traducir_regla_dominante(
            {"gbest_resultado": resultado},
            parametros_normalizacion,
            mapas,
            columna_consecuente
        )

        filas.append({
            "posicion": posicion,
            "regla": traduccion["texto_regla"],
            "consecuente": mapas[columna_consecuente][
                elemento["valor_consecuente"]
            ],
            "metodo_similitud": elemento["metodo_similitud"],
            "variables_antecedente": ", ".join(
                obtener_firma_variables(resultado)
            ),
            "numero_condiciones": metricas["numero_condiciones"],
            "fitness": resultado["fitness"],
            "soporte_XY": metricas["soporte_XY"],
            "confianza": metricas["confianza"],
            "lift": metricas["lift"],
            "factor_certeza": metricas["factor_certeza"],
            "miembros_familia_jaccard": elemento[
                "miembros_familia_jaccard"
            ],
            "numero_semillas": len(elemento["semillas"]),
            "numero_tamanos_enjambre": len(
                elemento.get("tamanos_enjambre", set())
            ),
            "tamanos_enjambre": ", ".join(
                str(tamano)
                for tamano in sorted(
                    elemento.get("tamanos_enjambre", set())
                )
            ),
            "semillas": ", ".join(
                str(semilla) for semilla in sorted(elemento["semillas"])
            )
        })

    return {
        "reglas_no_redundantes": reglas_no_redundantes,
        "tabla_reglas": pd.DataFrame(filas),
        "tabla_descartadas": pd.DataFrame(reglas_descartadas),
        "numero_grupos": len(grupos),
        "numero_comparaciones": comparaciones
    }


UMBRAL_JACCARD_FINAL = 0.85

resultado_filtro_jaccard = filtrar_reglas_por_jaccard(
    reglas_consolidadas=reglas_dominantes_consolidadas,
    dataframe=X_datos,
    parametros_normalizacion=parametros_normalizacion_robusta,
    mapas=mapas_categorias,
    columna_consecuente=columna_objetivo,
    umbral_jaccard=UMBRAL_JACCARD_FINAL
)

reglas_dominantes_finales = resultado_filtro_jaccard[
    "reglas_no_redundantes"
]
tabla_reglas_dominantes_finales = resultado_filtro_jaccard[
    "tabla_reglas"
]

print(
    f"Grupos estructurales: {resultado_filtro_jaccard['numero_grupos']}"
    f"\nReglas antes del filtro: {len(reglas_dominantes_consolidadas)}"
    f"\nReglas después del filtro: {len(reglas_dominantes_finales)}"
    f"\nReglas agrupadas: "
    f"{len(reglas_dominantes_consolidadas) - len(reglas_dominantes_finales)}"
    f"\nComparaciones Jaccard: "
    f"{resultado_filtro_jaccard['numero_comparaciones']}"
)

display(
    tabla_reglas_dominantes_finales.head(5).round({
        "fitness": 8,
        "soporte_XY": 6,
        "confianza": 6,
        "lift": 6,
        "factor_certeza": 6
    })
)

## Bloque 27. Analizar estabilidad de las reglas
La estabilidad se medirá según el número de semillas en las que apareció cada familia Jaccard:
alta: apareció en todas las semillas;
media: apareció en al menos dos;
exploratoria: apareció en una sola.

In [ ]:
# ============================================================
# ANALIZAR ESTABILIDAD DE LAS REGLAS
# ============================================================

tabla_estabilidad_reglas = (
    tabla_reglas_dominantes_finales.copy()
)

tabla_estabilidad_reglas.insert(
    0,
    "regla_id",
    [
        f"RD_{indice:04d}"
        for indice in range(
            1,
            len(tabla_estabilidad_reglas) + 1
        )
    ]
)

numero_total_semillas = len(SEMILLAS_EXPERIMENTO)

tabla_estabilidad_reglas["proporcion_semillas"] = (
    tabla_estabilidad_reglas["numero_semillas"]
    / numero_total_semillas
)


def clasificar_estabilidad(numero_semillas):
    if numero_semillas == numero_total_semillas:
        return "alta"

    if numero_semillas >= 2:
        return "media"

    return "exploratoria"


tabla_estabilidad_reglas["nivel_estabilidad"] = (
    tabla_estabilidad_reglas["numero_semillas"]
    .apply(clasificar_estabilidad)
)

tabla_estabilidad_reglas["es_reproducible"] = (
    tabla_estabilidad_reglas["numero_semillas"] >= 2
)

orden_estabilidad = {
    "alta": 0,
    "media": 1,
    "exploratoria": 2
}

tabla_estabilidad_reglas["orden_estabilidad"] = (
    tabla_estabilidad_reglas["nivel_estabilidad"]
    .map(orden_estabilidad)
)

tabla_reglas_ordenadas_estabilidad = (
    tabla_estabilidad_reglas
    .sort_values(
        by=[
            "orden_estabilidad",
            "miembros_familia_jaccard",
            "fitness"
        ],
        ascending=[True, False, False]
    )
    .drop(columns="orden_estabilidad")
    .reset_index(drop=True)
)

resumen_estabilidad = (
    tabla_estabilidad_reglas
    .groupby("nivel_estabilidad")
    .agg(
        cantidad_reglas=("regla_id", "count"),
        fitness_promedio=("fitness", "mean"),
        confianza_promedio=("confianza", "mean"),
        familias_representadas=(
            "miembros_familia_jaccard",
            "sum"
        )
    )
    .reset_index()
)

resumen_estabilidad["orden"] = (
    resumen_estabilidad["nivel_estabilidad"]
    .map(orden_estabilidad)
)

resumen_estabilidad = (
    resumen_estabilidad
    .sort_values("orden")
    .drop(columns="orden")
    .reset_index(drop=True)
)

print(f"Reglas finales: {len(tabla_estabilidad_reglas)}")
print(
    "Reglas reproducibles en al menos dos semillas: "
    f"{tabla_estabilidad_reglas['es_reproducible'].sum()}"
)
print("\nResumen de estabilidad:")

display(
    resumen_estabilidad.round({
        "fitness_promedio": 6,
        "confianza_promedio": 6
    })
)

print("Primeras reglas ordenadas por estabilidad:")

display(
    tabla_reglas_ordenadas_estabilidad[
        [
            "regla_id",
            "regla",
            "consecuente",
            "metodo_similitud",
            "fitness",
            "confianza",
            "lift",
            "numero_semillas",
            "nivel_estabilidad"
        ]
    ].head(5).round({
        "fitness": 8,
        "confianza": 6,
        "lift": 6
    })
)

## Catálogos de reglas dominantes

Se conservarán dos catálogos:

1. **Catálogo completo:** contiene todas las reglas dominantes no redundantes.
2. **Catálogo reproducible:** contiene las reglas encontradas en al menos dos semillas.

El catálogo reproducible será el conjunto principal para buscar reglas anómalas, mientras que el catálogo completo quedará disponible para análisis exploratorios.

In [ ]:
# ============================================================
# CREAR CATÁLOGOS DE REGLAS DOMINANTES
# ============================================================

catalogo_dominantes_completo = (
    tabla_estabilidad_reglas
    .copy()
    .sort_values(
        by=["consecuente", "fitness"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

catalogo_dominantes_reproducible = (
    tabla_estabilidad_reglas[
        tabla_estabilidad_reglas["es_reproducible"]
    ]
    .copy()
    .sort_values(
        by=[
            "numero_semillas",
            "miembros_familia_jaccard",
            "fitness"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

catalogo_dominantes_reproducible.insert(
    0,
    "dominante_id",
    [
        f"RD_REP_{indice:03d}"
        for indice in range(
            1,
            len(catalogo_dominantes_reproducible) + 1
        )
    ]
)

elementos_por_id = {
    f"RD_{indice:04d}": copy.deepcopy(elemento)
    for indice, elemento in enumerate(
        reglas_dominantes_finales,
        start=1
    )
}

reglas_dominantes_reproducibles = []

for _, fila in catalogo_dominantes_reproducible.iterrows():
    regla_id = fila["regla_id"]

    if regla_id not in elementos_por_id:
        raise ValueError(
            f"No se encontró la estructura de la regla {regla_id}."
        )

    elemento = elementos_por_id[regla_id]
    elemento["regla_id"] = regla_id
    elemento["dominante_id"] = fila["dominante_id"]

    reglas_dominantes_reproducibles.append(elemento)

if len(catalogo_dominantes_completo) != len(
    reglas_dominantes_finales
):
    raise ValueError(
        "El catálogo completo y las estructuras finales "
        "no tienen la misma cantidad de reglas."
    )

if len(catalogo_dominantes_reproducible) != len(
    reglas_dominantes_reproducibles
):
    raise ValueError(
        "El catálogo reproducible y sus estructuras internas "
        "no tienen la misma cantidad de reglas."
    )

print(
    f"Catálogo completo: "
    f"{len(catalogo_dominantes_completo)} reglas"
)
print(
    f"Catálogo reproducible: "
    f"{len(catalogo_dominantes_reproducible)} reglas"
)

display(
    catalogo_dominantes_reproducible[
        [
            "dominante_id",
            "regla",
            "consecuente",
            "metodo_similitud",
            "fitness",
            "confianza",
            "lift",
            "numero_semillas",
            "numero_tamanos_enjambre",
            "nivel_estabilidad"
        ]
    ].head(5).round({
        "fitness": 8,
        "confianza": 6,
        "lift": 6
    })
)

## Guardado de resultados

Se guardarán los catálogos de reglas, los resúmenes de estabilidad y una tabla comparativa entre los tamaños de enjambre.

El catálogo reproducible será el conjunto principal de reglas dominantes para la siguiente etapa.

In [ ]:
# ============================================================
# GUARDAR EXPERIMENTO COMPARATIVO
# ============================================================

carpeta_base = raiz_repositorio / "artifacts" / "dominante"
carpeta_destino = os.path.join(carpeta_base, NOMBRE_EXPERIMENTO)
os.makedirs(carpeta_destino, exist_ok=True)

if catalogo_dominantes_completo.empty:
    raise ValueError("El catalogo completo esta vacio.")

if catalogo_dominantes_reproducible.empty:
    raise ValueError("El catalogo reproducible esta vacio.")

cantidad_pbest = len(reglas_dominantes_multisemilla)
cantidad_consolidada = len(reglas_dominantes_consolidadas)
cantidad_final = len(catalogo_dominantes_completo)
cantidad_reproducible = len(catalogo_dominantes_reproducible)

maximo_pbest = (
    sum(
        tamano["numero_particulas"]
        for tamano in TAMANOS_ENJAMBRE
    )
    * len(configuraciones_experimento)
    * len(SEMILLAS_EXPERIMENTO)
)

evaluaciones_particulas = (
    sum(
        tamano["numero_particulas"]
        * (tamano["numero_iteraciones"] + 1)
        for tamano in TAMANOS_ENJAMBRE
    )
    * len(configuraciones_experimento)
    * len(SEMILLAS_EXPERIMENTO)
)

conteo_estabilidad = (
    catalogo_dominantes_completo["nivel_estabilidad"]
    .value_counts()
)

resumen_experimento = pd.DataFrame([{
    "experimento": NOMBRE_EXPERIMENTO,
    "tamanos_enjambre": ", ".join(
        str(tamano["numero_particulas"])
        for tamano in TAMANOS_ENJAMBRE
    ),
    "numero_tamanos_enjambre": len(TAMANOS_ENJAMBRE),
    "numero_iteraciones_por_tamano": (
        NUMERO_ITERACIONES_EXPERIMENTO
    ),
    "numero_semillas": len(SEMILLAS_EXPERIMENTO),
    "numero_configuraciones": len(configuraciones_experimento),
    "total_ejecuciones": total_ejecuciones,
    "evaluaciones_particulas": evaluaciones_particulas,
    "maximo_pbest": maximo_pbest,
    "reglas_dominantes_pbest": cantidad_pbest,
    "tasa_dominancia_pbest": cantidad_pbest / maximo_pbest,
    "reglas_consolidadas": cantidad_consolidada,
    "duplicados_parametricos": (
        cantidad_pbest - cantidad_consolidada
    ),
    "reglas_finales": cantidad_final,
    "reglas_agrupadas_jaccard": (
        cantidad_consolidada - cantidad_final
    ),
    "reglas_reproducibles": cantidad_reproducible,
    "proporcion_reproducibles": (
        cantidad_reproducible / cantidad_final
    ),
    "estabilidad_alta": int(
        conteo_estabilidad.get("alta", 0)
    ),
    "estabilidad_media": int(
        conteo_estabilidad.get("media", 0)
    ),
    "estabilidad_exploratoria": int(
        conteo_estabilidad.get("exploratoria", 0)
    ),
    "soporte_minimo": SOPORTE_MINIMO_DOMINANTE,
    "confianza_minima": CONFIANZA_MINIMA_DOMINANTE,
    "lift_minimo": LIFT_MINIMO_DOMINANTE,
    "factor_certeza_minimo": CF_MINIMO_DOMINANTE,
    "umbral_jaccard": UMBRAL_JACCARD_FINAL
}])

resultados_reglas_dominantes = {
    "nombre_experimento": NOMBRE_EXPERIMENTO,
    "reglas_dominantes_reproducibles": (
        reglas_dominantes_reproducibles
    ),
    "reglas_dominantes_finales": reglas_dominantes_finales,
    "catalogo_completo": catalogo_dominantes_completo,
    "catalogo_reproducible": (
        catalogo_dominantes_reproducible
    ),
    "resumen_experimento": resumen_experimento,
    "resumen_ejecuciones": tabla_resumen_multisemilla,
    "resumen_tamanos_enjambre": (
        resumen_tamanos_enjambre
    ),
    "resumen_estabilidad": resumen_estabilidad,
    "criterios_dominancia": {
        "soporte_minimo": SOPORTE_MINIMO_DOMINANTE,
        "confianza_minima": CONFIANZA_MINIMA_DOMINANTE,
        "lift_minimo": LIFT_MINIMO_DOMINANTE,
        "factor_certeza_minimo": CF_MINIMO_DOMINANTE,
        "min_condiciones": MIN_CONDICIONES,
        "max_condiciones": MAX_CONDICIONES
    },
    "configuracion_pso": {
        "tamanos_enjambre": TAMANOS_ENJAMBRE,
        "semillas": SEMILLAS_EXPERIMENTO,
        "configuraciones": (
            configuraciones_experimento
        ),
        "probabilidad_exploracion": (
            PROBABILIDAD_EXPLORACION_EXPERIMENTO
        ),
        "radio_minimo": RADIO_MINIMO,
        "radio_maximo": RADIO_MAXIMO,
        "w_inercia": W_INERCIA,
        "c1_cognitivo": C1_COGNITIVO,
        "c2_social": C2_SOCIAL,
        "velocidad_max_activacion": (
            VELOCIDAD_MAX_ACTIVACION
        ),
        "velocidad_max_centro": (
            VELOCIDAD_MAX_CENTRO
        ),
        "velocidad_max_radio": (
            VELOCIDAD_MAX_RADIO
        )
    },
    "criterio_redundancia": {
        "metodo": "Jaccard difuso",
        "umbral_jaccard": UMBRAL_JACCARD_FINAL,
        "numero_grupos": (
            resultado_filtro_jaccard["numero_grupos"]
        ),
        "numero_comparaciones": (
            resultado_filtro_jaccard[
                "numero_comparaciones"
            ]
        )
    },
    "columnas_continuas": columnas_continuas,
    "columnas_categoricas": columnas_categoricas,
    "columna_objetivo": columna_objetivo,
    "mapas_categorias": mapas_categorias,
    "parametros_normalizacion": (
        parametros_normalizacion_robusta
    )
}

rutas_salida = {
    "catalogo_completo": os.path.join(
        carpeta_destino,
        "catalogo_reglas_dominantes_completo.csv"
    ),
    "catalogo_reproducible": os.path.join(
        carpeta_destino,
        "catalogo_reglas_dominantes_reproducible.csv"
    ),
    "resumen_multisemilla": os.path.join(
        carpeta_destino,
        "resumen_ejecuciones_reglas_dominantes.csv"
    ),
    "resumen_tamanos_enjambre": os.path.join(
        carpeta_destino,
        "resumen_tamanos_enjambre.csv"
    ),
    "resumen_estabilidad": os.path.join(
        carpeta_destino,
        "resumen_estabilidad_reglas.csv"
    ),
    "resumen_experimento": os.path.join(
        carpeta_destino,
        "resumen_experimento.csv"
    ),
    "resultados": os.path.join(
        carpeta_destino,
        "resultados_reglas_dominantes.pkl"
    )
}

catalogo_dominantes_completo.to_csv(
    rutas_salida["catalogo_completo"],
    index=False
)

catalogo_dominantes_reproducible.to_csv(
    rutas_salida["catalogo_reproducible"],
    index=False
)

tabla_resumen_multisemilla.to_csv(
    rutas_salida["resumen_multisemilla"],
    index=False
)

resumen_tamanos_enjambre.to_csv(
    rutas_salida["resumen_tamanos_enjambre"],
    index=False
)

resumen_estabilidad.to_csv(
    rutas_salida["resumen_estabilidad"],
    index=False
)

resumen_experimento.to_csv(
    rutas_salida["resumen_experimento"],
    index=False
)

joblib.dump(
    resultados_reglas_dominantes,
    rutas_salida["resultados"]
)

print(
    f"Resultados guardados en: {carpeta_destino}"
    f"\nEjecuciones: {total_ejecuciones}"
    f"\nReglas dominantes pBest: {cantidad_pbest}"
    f"\nReglas consolidadas: {cantidad_consolidada}"
    f"\nReglas finales: {cantidad_final}"
    f"\nReglas reproducibles: {cantidad_reproducible}"
)

display(resumen_experimento.round(6))
